# Visualization of contact maps in *A. rouxii* using cooltools tutorial

## Import libraries

In [ ]:
# Import standard python libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Import cooltools and cooler
import cooltools
import cooler

In [ ]:
# Import libraries for plotting
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cooltools.lib.plotting

from matplotlib.ticker import EngFormatter # For adding bp to plots
import matplotlib.patches as patches # For adding circles to the plots

## Set format for file

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

## Check available resolutions and load mcool file

In [ ]:
# Resolutions in mcool file
cooler.fileops.list_coolers('/Users/emma/Documents/NMBU/Master/Master/Koding_python/ggAmyRoux1.mcool')

In [ ]:
# Load the cool file at 80 kb resolution
res_80k = cooler.Cooler('/Users/emma/Documents/NMBU/Master/Master/AmyRoux/ggAmyRoux1.mcool::resolutions/80000')
resolution = res_80k.binsize
print(resolution)

## Inspect the mcool file

In [ ]:
# to print chromosome names and binsize for this cooler
print(f'chromosomes: {res_80k.chromnames}, binsize: {res_80k.binsize}')

In [ ]:
# Makes a list of chromosomes and binsize present in this cooler file. 
chromstarts = []
for i in res_80k.chromnames:
    print(f'{i} : {res_80k.extent(i)}')
    chromstarts.append(res_80k.extent(i)[0])

In [ ]:
# Makes a list of chromosomes and binsize present in this cooler file. 
chromstarts = []
for i in res_80k.chromnames:
    print(f'{i} : {res_80k.extent(i)}')
    chromstarts.append(res_80k.extent(i)[0])

## Filtering out scaffolds

In [ ]:
# Filtering out scaffolds
min_size = 100_000

# A. rouxii has SUPER's and scaffolds, all assemblies named "scaffold" are removed independently of their size. 
# If they are smaller than 100 000 are also removed. 
chroms_to_keep = [c for c in res_80k.chromnames if res_80k.chromsizes[c] >= min_size and "Scaffold" not in c]

print("Chromosomes kept:", chroms_to_keep) # Print the chromosomes that we plot. 

In [ ]:
# Get the bins for the chromosomes we keep
bins_res_80k = res_80k.bins()[:]
keep_mask_res80k = bins_res_80k['chrom'].isin(chroms_to_keep)

In [ ]:
# Fetch full genome-wide matrix (
full_matrix_res80k = res_80k.matrix(balance=False)[:]

# Subset matrix to only the filtered chromosomes
matrix_filtered_res80k = full_matrix_res80k[keep_mask_res80k.values, :][:, keep_mask_res80k.values]

In [ ]:
# Shorten the chromosome names so that the plot only shows the chromosome number
chromstarts = []
pos = 0
short_labels = []
for c in chroms_to_keep:
    n_bins = (bins_res_80k[bins_res_80k['chrom'] == c].shape[0])
    chromstarts.append(pos)
    short_labels.append(c.split('_')[-1])
    pos += n_bins


# Visualizing the Hi-C data

### Genome-wide plot of *A. rouxii*

In [ ]:
# Plot
f, ax = plt.subplots(figsize=(8, 8))
norm = LogNorm(vmin=1, vmax=1000)

im = ax.matshow(matrix_filtered_res80k, cmap='fall', norm=norm) # Plott filtered genome-wide contact map with fall colours

# Colorbar settings
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Contact counts (log scale)', fontsize=16)
cbar.ax.tick_params(labelsize=12)

# Set the x/y-axis number at the middle of the chromosome
chromcenters = []
for i in range(len(chromstarts)):
    if i < len(chromstarts) - 1:
        chromcenters.append((chromstarts[i] + chromstarts[i+1]) / 2)
    else:
        chromcenters.append((chromstarts[i] + pos) / 2)

# Add ticks and labels to the x- and y-axis
ax.set_xticks(chromcenters)
ax.set_yticks(chromcenters)
ax.set_xticklabels(short_labels, fontsize=11)
ax.set_yticklabels(short_labels, fontsize=11)
ax.xaxis.tick_bottom()
ax.set_title(f"A. rouxii (80 k)", fontsize=20)

ax.set_xlabel("Chromosome", fontsize=16, labelpad=10)
ax.set_ylabel("Chromosome", fontsize=16, labelpad=10)

# Adds grey lines at chromosome boundaries 
for s in chromstarts:
    ax.axhline(s - 0.5, color='grey', lw=0.5)
    ax.axvline(s - 0.5, color='grey', lw=0.5)


plt.tight_layout()
plt.show()


## Functions used for plotting (found in the cooltools tutorial)

In [ ]:
# Function for adding basepairs to plots
bp_formatter = EngFormatter('b') # Here used to formate to get genomic coordinates in megabases
def format_ticks(ax, x=True, y=True, rotate=True): # To format axis ticks 
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_top()  # move x ticks to top for genome plots
    if rotate:
        ax.tick_params(axis='x')

## Predicted centromeres in *A. rouxii*

In [ ]:
# Predicted centromeres in A. rouxii
centromeres = {
    "subgenom1_SUPER_1": (3011, 62819),
    "subgenom1_SUPER_2": (3357007, 3370569), # Maybe 
    "subgenom1_SUPER_3": (981857, 998763),
    "subgenom1_SUPER_4": (2697696, 2699846),
    "subgenom1_SUPER_5": (3509447, 3535106),
    "subgenom1_SUPER_6": (3359543, 3382640),
    "subgenom1_SUPER_7": (2461517, 2920085),
    "subgenom1_SUPER_8": (1904802, 1905129),
    "subgenom1_SUPER_10": (72483, 88619),
    "subgenom2_SUPER_1": (110102, 115682), # Maybe
    "subgenom2_SUPER_2": (3364087, 3410879),
    "subgenom2_SUPER_4": (2458454, 2481981), # Maybe
    "subgenom2_SUPER_5": (3264106, 3498655), #Maybe
    "subgenom2_SUPER_6": (317804, 1096308), #mb
    "subgenom2_SUPER_7": (2636269, 2661678), #maybe
    "subgenom2_SUPER_8": (1173249, 1195961), #maybe
    "subgenom2_SUPER_9": (706583, 1652021), #maybe
}

## Import mcool at higher resolution

In [ ]:
# Import mcool file at 40 kb resolution
res_40k = cooler.Cooler('/Users/emma/Documents/NMBU/Master/Master/AmyRoux/ggAmyRoux1.mcool::resolutions/40000') 

# The resolution of contact maps
resolution = res_40k.binsize // 1000 
print(resolution)

## For loop going through all chromosomes with predicted centromere

In [ ]:
# Loop over all chromosomes with predicted centromeres 
# Loop goes through all the annotated centromeres and plot the contact map for the given chromosome together with the centromere annotation.
for chrom, (cent_start, cent_end) in centromeres.items():

    # Full chromosome coordinates
    start = 0 # all chromosomes start with 0
    end = res_40k.chromsizes[chrom] # end is the chromosome size
    region = f"{chrom}:{start}-{end}" # plot the whole chromosome
    extents = (start, end, end, start)

    # Chromosome label from short_labels
    chrom_index = chroms_to_keep.index(chrom) # Uses chroms_to_keep to label chromosomes 
    chrom_label = short_labels[chrom_index] # Use short_labels to shorten the chromosomes name in chroms_to_keep

    # Fetch matrix for the region (the whole chromosome)
    matrix = res_40k.matrix(balance=False).fetch(region)

    # Plot
    f, ax = plt.subplots(figsize=(8, 8)) # plot size
    norm = LogNorm(vmin=1, vmax=1000)
    im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Contact counts (log scale)', fontsize=14)

    # Axis labels
    ax.set_xlabel('Genomic position', fontsize = 14)
    ax.set_ylabel('Genomic position', fontsize = 14)

    # Use format_ticks to get basepairs along the x- and y-axis
    format_ticks(ax)

   # Draw a circle around the centromere 
    cent_mid = (cent_start + cent_end) / 2 # Find the middle of the centromere interval
    radius = 0.03 * (end - start)  # The radius of the circle is 3% of chromosome length
    circle = patches.Circle((cent_mid, cent_mid), radius=radius, # Draw a black circle around the centromere
                            edgecolor='black', facecolor='none', lw=2)
    ax.add_patch(circle) # Add the circle to the plot

    # Plot Title (Chromosome number and resolution)
    ax.set_title(f'Chromosome {chrom_label} in A. rouxii ({resolution} k)', fontsize = 16)

    plt.tight_layout()
    plt.show()


## Loop through all chromosomes

In [ ]:
# Loop over all chromosomes that are part of filtering regardless of centromere prediction
for chrom in chroms_to_keep:

    # Full chromosome coordinates
    start = 0
    end = res_40k.chromsizes[chrom]
    region = f"{chrom}:{start}-{end}"
    extents = (start, end, end, start)

    # Get short chromosome label
    chrom_index = chroms_to_keep.index(chrom)
    chrom_label = short_labels[chrom_index]

    # Fetch Hi-C matrix
    matrix = res_40k.matrix(balance=False).fetch(region)

    # Plot
    f, ax = plt.subplots(figsize=(8, 8))
    norm = LogNorm(vmin=1, vmax=1000)
    im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Contact counts (log scale)', fontsize=14)

    # Axis labels
    ax.set_xlabel('Genomic position', fontsize = 14)
    ax.set_ylabel('Genomic position', fontsize = 14)

    format_ticks(ax)

    # Title
    ax.set_title(f'Chromosome {chrom_label} in A. rouxii ({resolution} k)', fontsize = 16)

    plt.tight_layout()
    plt.show()



### Without SUPER_3

In [ ]:
chroms_to_keep = [
    c for c in res_80k.chromnames
    if res_80k.chromsizes[c] >= min_size and
       "subgenom1_SUPER_3" not in c and
       "subgenom2_SUPER_3" not in c and
       "Scaffold" not in c
]
print("Chromosomes kept:", chroms_to_keep) # Print the chromosomes that we plot. 

In [ ]:
# Get the bins for the chromosomes we keep
bins_res_80k = res_80k.bins()[:]
keep_mask_res80k = bins_res_80k['chrom'].isin(chroms_to_keep)
# Fetch full genome-wide matrix (balanced or raw)
full_matrix_res80k = res_80k.matrix(balance=False)[:]

# Subset matrix to only keep large chromosomes
matrix_filtered_res80k = full_matrix_res80k[keep_mask_res80k.values, :][:, keep_mask_res80k.values]

In [ ]:
# Shorten the chromosome names so that the plot only shows the chromosome number
labels = res_80k.chromnames  # Names like ['SUPER_1', 'SUPER_2', 'SUPER_3', ...]
# Convert them to just the numbers
short_labels = [lbl.split('_')[-1] for lbl in labels] # Names like 1,2,3, ...
# Short labels for ticks
chromstarts = []
pos = 0
short_labels = []
for c in chroms_to_keep:
    n_bins = (bins_res_80k[bins_res_80k['chrom'] == c].shape[0])
    chromstarts.append(pos)
    short_labels.append(c.split('_')[-1])
    pos += n_bins


In [ ]:
# Plot
f, ax = plt.subplots(figsize=(8, 8))
norm = LogNorm(vmin=1, vmax=1000)

im = ax.matshow(matrix_filtered_res80k, cmap='fall', norm=norm)

# color bar settings
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Contact counts (log scale)', fontsize=12)
cbar.ax.tick_params(labelsize=9)

# Set the x/y-axis number at the middle of the chromosome
chromcenters = []
for i in range(len(chromstarts)):
    if i < len(chromstarts) - 1:
        chromcenters.append((chromstarts[i] + chromstarts[i+1]) / 2)
    else:
        chromcenters.append((chromstarts[i] + pos) / 2)

# Add ticks and labels to the x- and y-axis
ax.set_xticks(chromcenters)
ax.set_yticks(chromcenters)
ax.set_xticklabels(short_labels, fontsize=11)
ax.set_yticklabels(short_labels, fontsize=11)
ax.xaxis.tick_bottom()
ax.set_title(f"A. rouxii (80 k)", fontsize=16)

ax.set_xlabel("Chromosome", fontsize=12, labelpad=10)
ax.set_ylabel("Chromosome", fontsize=12, labelpad=10)

# Adds grey lines at chromosome boundaries 
for s in chromstarts:
    ax.axhline(s - 0.5, color='grey', lw=0.5)
    ax.axvline(s - 0.5, color='grey', lw=0.5)


plt.tight_layout()
plt.show()
